In [4]:
import pandas as pd

group_df = pd.read_csv("../data/group_fixtures.csv")
knockout_df = pd.read_csv("../data/knockout_slots.csv")

group_df.head()

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [5]:
 knockout_df.head()

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I


In [6]:
print("Group Matches:", len(group_df))
print("Knockout Matches:", len(knockout_df))

print("Unique groups:", group_df["group"].nunique())
print("Unique teams (raw):", len(set(group_df["home_team"]).union(set(group_df["away_team"]))))

Group Matches: 72
Knockout Matches: 32
Unique groups: 12
Unique teams (raw): 48


In [7]:
# BUILD GROUP STRUCTURE
groups = {}

for g in sorted(group_df["group"].unique()):
    teams = set()

    temp = group_df[group_df["group"] == g]

    for _, row in temp.iterrows():
        teams.add(row["home_team"])
        teams.add(row["away_team"])

    groups[g] = sorted(list(teams))

groups

{'A': ['Mexico', 'South Africa', 'South Korea', 'UEFA Playoff D'],
 'B': ['Canada', 'Qatar', 'Switzerland', 'UEFA Playoff A'],
 'C': ['Brazil', 'Haiti', 'Morocco', 'Scotland'],
 'D': ['Australia', 'Paraguay', 'UEFA Playoff C', 'USA'],
 'E': ['Curaçao', "Côte d'Ivoire", 'Ecuador', 'Germany'],
 'F': ['Japan', 'Netherlands', 'Tunisia', 'UEFA Playoff B'],
 'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
 'H': ['Cabo Verde', 'Saudi Arabia', 'Spain', 'Uruguay'],
 'I': ['FIFA Playoff 2', 'France', 'Norway', 'Senegal'],
 'J': ['Algeria', 'Argentina', 'Austria', 'Jordan'],
 'K': ['Colombia', 'FIFA Playoff 1', 'Portugal', 'Uzbekistan'],
 'L': ['Croatia', 'England', 'Ghana', 'Panama']}

In [8]:
# VALIDATION OUTPUT
for g, teams in groups.items():
    print(f"Group {g} -> {len(teams)} teams")

Group A -> 4 teams
Group B -> 4 teams
Group C -> 4 teams
Group D -> 4 teams
Group E -> 4 teams
Group F -> 4 teams
Group G -> 4 teams
Group H -> 4 teams
Group I -> 4 teams
Group J -> 4 teams
Group K -> 4 teams
Group L -> 4 teams


In [11]:
# CHECK TOTAL TEAMS
all_teams = set()

for teams in groups.values():
    all_teams.update(teams)

len(all_teams), sorted(list(all_teams))

(48,
 ['Algeria',
  'Argentina',
  'Australia',
  'Austria',
  'Belgium',
  'Brazil',
  'Cabo Verde',
  'Canada',
  'Colombia',
  'Croatia',
  'Curaçao',
  "Côte d'Ivoire",
  'Ecuador',
  'Egypt',
  'England',
  'FIFA Playoff 1',
  'FIFA Playoff 2',
  'France',
  'Germany',
  'Ghana',
  'Haiti',
  'Iran',
  'Japan',
  'Jordan',
  'Mexico',
  'Morocco',
  'Netherlands',
  'New Zealand',
  'Norway',
  'Panama',
  'Paraguay',
  'Portugal',
  'Qatar',
  'Saudi Arabia',
  'Scotland',
  'Senegal',
  'South Africa',
  'South Korea',
  'Spain',
  'Switzerland',
  'Tunisia',
  'UEFA Playoff A',
  'UEFA Playoff B',
  'UEFA Playoff C',
  'UEFA Playoff D',
  'USA',
  'Uruguay',
  'Uzbekistan'])

In [12]:
# SLOT STRUCTURE 
knockout_map = {}

for _, row in knockout_df.iterrows():
    knockout_map[row["match_id"]] = {
        "round": row["round"],
        "home_slot": row["slot_home"],
        "away_slot": row["slot_away"],
        "multiplier": row["multiplier"]
    }

list(knockout_map.items())[:2]

[(73,
  {'round': 'Round of 32',
   'home_slot': 'Runner-up Group A',
   'away_slot': 'Runner-up Group B',
   'multiplier': 1}),
 (74,
  {'round': 'Round of 32',
   'home_slot': 'Winner Group C',
   'away_slot': 'Runner-up Group F',
   'multiplier': 1})]

In [13]:
# SLOT TYPE ANALYSIS 
from collections import Counter

slot_types = []

for row in knockout_df.itertuples():
    slot_types.append(row.slot_home)
    slot_types.append(row.slot_away)

Counter(slot_types).most_common(20)

[('Runner-up Group A', 1),
 ('Runner-up Group B', 1),
 ('Winner Group C', 1),
 ('Runner-up Group F', 1),
 ('Winner Group E', 1),
 ('Best 3rd (Groups A/B/C/D/F)', 1),
 ('Winner Group F', 1),
 ('Runner-up Group C', 1),
 ('Runner-up Group E', 1),
 ('Runner-up Group I', 1),
 ('Winner Group I', 1),
 ('Best 3rd (Groups C/D/F/G/H)', 1),
 ('Winner Group A', 1),
 ('Best 3rd (Groups C/E/F/H/I)', 1),
 ('Winner Group L', 1),
 ('Best 3rd (Groups E/H/I/J/K)', 1),
 ('Winner Group G', 1),
 ('Best 3rd (Groups A/E/H/I/J)', 1),
 ('Winner Group D', 1),
 ('Best 3rd (Groups B/E/F/I/J)', 1)]

In [14]:
# CRITICAL VALIDATION RULE 
missing_groups = []

for g in "ABCDEFGHIJKL":
    if g not in groups:
        missing_groups.append(g)

missing_groups

[]

In [17]:
knockout_map[73]


{'round': 'Round of 32',
 'home_slot': 'Runner-up Group A',
 'away_slot': 'Runner-up Group B',
 'multiplier': 1}